In [1]:
"""Generate rutile (110) CTR surface cells from PyXtal Wyckoff metadata.

This example shows the intended workflow for symmetry-aware CTR models:

1. start from a parent crystal structure,
2. enumerate distinct bulk terminations using a matrix or Miller indices,
3. build a candidate, or preserve an existing fixed-origin surface cell,
4. inspect couplings and optionally add a symmetry-preserving fit parameter.

The generated atoms are normal CTR atoms. Symmetry is metadata until a user
explicitly adds a Wyckoff parameter.
"""

import sys
from dataclasses import replace
from pathlib import Path

import numpy as np

REPOSITORY_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "orgui").is_dir()
)
sys.path.insert(0, str(REPOSITORY_ROOT))

from pymatgen.core import Lattice, Structure  # noqa: E402

from orgui.datautils.xrayutils import CTRsymmetry  # noqa: E402


def rutile_parent_structure(metal, parent_a, oxygen_u, dw=None):
    """Create a conventional rutile parent structure.

    :param str metal:
        Metal element symbol, for example ``"Ru"`` or ``"Ti"``.
    :param tuple parent_a:
        Parent conventional rutile lattice constants in Angstrom.
    :param float oxygen_u:
        Oxygen Wyckoff ``4f`` internal coordinate.
    :param dw:
        Optional isotropic B parameter in Angstrom squared, stored on each
        source site. None leaves displacement data unspecified.
    :returns:
        Pymatgen structure accepted by PyXtal.
    :rtype:
        pymatgen.core.Structure
    """
    u = oxygen_u
    return Structure(
        Lattice.tetragonal(parent_a[0], parent_a[2]),
        [metal, metal, "O", "O", "O", "O"],
        [
            [0.0, 0.0, 0.0],
            [0.5, 0.5, 0.5],
            [u, u, 0.0],
            [1.0 - u, 1.0 - u, 0.0],
            [0.5 - u, 0.5 + u, 0.5],
            [0.5 + u, 0.5 - u, 0.5],
        ],
        site_properties={} if dw is None else {"B_iso": [dw] * 6},
    )


def make_rutile_110_surface(name, metal, parent_a, oxygen_u, dw):
    """Create the original fixed-origin rutile (110) ``UnitCell``.

    Keep the established z=0 cut and two-layer grouping for existing fits.
    ``surface_terminations`` below demonstrates midpoint cuts separately.

    :param str name:
        Unit-cell name.
    :param str metal:
        Metal element symbol, for example ``"Ru"`` or ``"Ti"``.
    :param tuple parent_a:
        Parent conventional rutile lattice constants in Angstrom.
    :param float oxygen_u:
        Oxygen Wyckoff ``4f`` internal coordinate.
    :param float dw:
        In-plane and out-of-plane B parameter in Angstrom squared.
    :returns:
        Generated CTR unit cell with attached symmetry metadata.
    :rtype:
        orgui.datautils.xrayutils.CTRuc.UnitCell
    """
    parent = rutile_parent_structure(metal, parent_a, oxygen_u, dw=dw)
    surface_spec = replace(
        CTRsymmetry.rutile_110_surface_spec(parent_a), translation_range=None,
    )
    return CTRsymmetry.surface_unitcell_from_seed(
        parent, surface_spec, name, tol=1e-3, fallback_B=None,
    )


def print_wyckoff_summary(unitcell):
    """Print generated sites and affine couplings.

    :param orgui.datautils.xrayutils.CTRuc.UnitCell unitcell:
        Generated unit cell with attached symmetry metadata.
    """
    print(f"\n{unitcell.name}")
    print("surface cell a/Angstrom:", [round(value, 5) for value in unitcell.a])
    print("atoms:", len(unitcell.names))

    for site in unitcell.wyckoff_sites():
        print(
            f"site {site['site_id']} ({site['wyckoff_label']}): "
            f"atoms={site['atom_indices']} status={site['status']}"
        )

    print("oxygen u couplings:")
    for coupling in unitcell.wyckoff_couplings("O_4f"):
        expression = f"{coupling.constant:+.5f} {coupling.factor:+.1f}*u"
        print(
            f"  atom {coupling.atom_index:02d} "
            f"{unitcell.names[coupling.atom_index]}.{coupling.coordinate} = "
            f"{expression}"
        )

In [2]:
"""Run the rutile (110) Wyckoff metadata example."""
parent = rutile_parent_structure(
    "Ru", (4.653255, 4.653255, 2.9692), 0.30569, dw=0.5033,
)
options = CTRsymmetry.surface_terminations(
    parent,
    # Right-handed candidate basis: A=[-1,1,0], B=[0,0,1], C=[1,1,0].
    transform=[[-1, 0, 1], [1, 0, 1], [0, 1, 0]],
    cell_mode="lattice",  # also the default for a full matrix
    layer_tolerance=0.25,  # Angstrom along the true surface normal
)
for option in options:
    print(option.id, "equivalent cuts:", option.equivalent_cuts)
    print("layers from surface inward:", [
        layer["composition"] for layer in option.layer_sequence
    ])
# First candidate is an illustration, not a predicted stable termination.
candidate = options[0].build_unitcell("RuO2_midpoint_cut")
print("Candidate z range (crystal below interface at zero):",
      candidate.basis[:, 3].min(), candidate.basis[:, 3].max())

# Miller indices default to c along the surface normal. The crystal repeat
# can have a lateral component, stored separately from this coordinate c.
automatic = CTRsymmetry.surface_terminations(
    parent, miller=(1, 1, 0), in_plane=(1, -1, 0),
)
print("Automatic surface basis (columns):", automatic[0].surface_spec.transform)
surface = automatic[0].build_unitcell("RuO2_surface")
print("Surface angles (degrees):", np.rad2deg(surface.alpha))
print("Crystal repeat (surface fractional):", surface.stacking_translation)
# Use cell_mode="lattice" with Miller indices to recover the oblique basis.

# Preserve the existing fixed-origin examples and their fit conventions.
ruo2 = make_rutile_110_surface(
    name="RuO2",
    metal="Ru",
    parent_a=(4.653255, 4.653255, 2.9692),
    oxygen_u=0.30569,
    dw=0.5033,
)
tio2 = make_rutile_110_surface(
    name="TiO2",
    metal="Ti",
    parent_a=(4.653255, 4.653255, 2.9692),
    oxygen_u=0.30458,
    dw=0.3719,
)

print_wyckoff_summary(ruo2)
print_wyckoff_summary(tio2)

ruo2.addWyckoffParameter(
    "O_4f",
    "u",
    absolute_limits=(0.28, 0.34),
)
print("\nAfter adding symmetry-preserving RuO2 oxygen u parameter:")
print(ruo2.wyckoff_sites()[1])
print("initial fit value is absolute u:", ruo2.getInitialParameters())

ruo2.setFitParameters([0.31569])
print("first few oxygen coordinates after delta_u = +0.01:")
oxygen_indices = sorted(
    {coupling.atom_index for coupling in ruo2.wyckoff_couplings("O_4f")}
)
for index in oxygen_indices[:4]:
    xyz = [round(value, 5) for value in ruo2.basis[index, 1:4]]
    print(index, ruo2.names[index], xyz)

tio2.addFitParameter(
    ([0], ["z"]),
    limits=(0.0, 1.0),
    name="manual_top_oxygen_z",
)
print("\nManual single-atom TiO2 fit intentionally lowers symmetry:")
print(tio2.wyckoff_sites()[1])

cut-000 equivalent cuts: ((0.0, 0.19430999999999998), (0.5, 0.69431))
layers from surface inward: [{'O': 2.0, 'Ru': 2.0}, {'O': 1.0}, {'O': 1.0}, {'O': 2.0, 'Ru': 2.0}, {'O': 1.0}, {'O': 1.0}]
cut-001 equivalent cuts: ((0.19430999999999998, 0.30569), (0.69431, 0.80569))
layers from surface inward: [{'O': 1.0}, {'O': 2.0, 'Ru': 2.0}, {'O': 1.0}, {'O': 1.0}, {'O': 2.0, 'Ru': 2.0}, {'O': 1.0}]
cut-002 equivalent cuts: ((0.30569, 0.5), (0.80569, 1.0))
layers from surface inward: [{'O': 1.0}, {'O': 1.0}, {'O': 2.0, 'Ru': 2.0}, {'O': 1.0}, {'O': 1.0}, {'O': 2.0, 'Ru': 2.0}]


Candidate z range (crystal below interface at zero): -0.9028449999999999 -0.09715499999999988
Automatic surface basis (columns): [[ 1.00000000e+00  0.00000000e+00  5.00000000e-01]
 [-1.00000000e+00  0.00000000e+00  5.00000000e-01]
 [ 0.00000000e+00 -1.00000000e+00 -9.59617715e-17]]
Surface angles (degrees): [90. 90. 90.]
Crystal repeat (surface fractional): (0.5, 0.0, 1.0)

RuO2
surface cell a/Angstrom: [np.float64(6.5807), np.float64(2.9692), np.float64(6.5807)]
atoms: 12
site Ru_2a (2a): atoms=[4, 5, 10, 11] status=metadata_only
site O_4f (4f): atoms=[0, 1, 2, 3, 6, 7, 8, 9] status=metadata_only
oxygen u couplings:
  atom 00 O.x = +0.00000 +1.0*u
  atom 01 O.x = +1.00000 -1.0*u
  atom 02 O.z = +1.00000 -1.0*u
  atom 03 O.z = +0.50000 +1.0*u
  atom 06 O.x = +0.50000 -1.0*u
  atom 07 O.x = +0.50000 +1.0*u
  atom 08 O.z = +0.50000 -1.0*u
  atom 09 O.z = +0.00000 +1.0*u

TiO2
surface cell a/Angstrom: [np.float64(6.5807), np.float64(2.9692), np.float64(6.5807)]
atoms: 12
site Ti_2a (2a): 

In [3]:
tio2.wyckoff_sites()

[{'site_id': 'Ti_2a',
  'element': 'Ti',
  'wyckoff_label': '2a',
  'variables': {},
  'occ': 1.0,
  'iDW': 0.3719,
  'oDW': 0.3719,
  'representative_parent_fractional': (0.0, 0.0, 0.0),
  'atom_indices': [4, 5, 10, 11],
  'spacegroup_number': 136,
  'spacegroup_symbol': 'P42/mnm',
  'status': 'metadata_only'},
 {'site_id': 'O_4f',
  'element': 'O',
  'wyckoff_label': '4f',
  'variables': {'u': 0.30458},
  'occ': 1.0,
  'iDW': 0.3719,
  'oDW': 0.3719,
  'representative_parent_fractional': (0.30458, 0.30458, 0.0),
  'atom_indices': [0, 1, 2, 3, 6, 7, 8, 9],
  'spacegroup_number': 136,
  'spacegroup_symbol': 'P42/mnm',
  'status': 'partially_overridden'}]

In [4]:
print(tio2)

Lattice:
Real space: 	[6.58069633 2.9692     6.58069633] / [90. 90. 90.]
Reciprocal space: 	[0.95479034 2.11612061 0.95479034] / [90. 90. 90.]
12 atoms in unit cell
id  Name        x/frac      y/frac      z/frac       iDW       oDW     occup  layerIdx
00  O          0.30458     0.50000     0.50000    0.3719    0.3719    1.0000         2
01  O          0.69542     0.50000     0.50000    0.3719    0.3719    1.0000         2
02  O          0.00000     0.00000     0.69542    0.3719    0.3719    1.0000         2
03  O          0.50000     0.00000     0.80458    0.3719    0.3719    1.0000         2
04  Ti         0.50000     0.00000     0.50000    0.3719    0.3719    1.0000         2
05  Ti         0.00000     0.50000     0.50000    0.3719    0.3719    1.0000         2
06  O          0.19542     0.50000     0.00000    0.3719    0.3719    1.0000         1
07  O          0.80458     0.50000     0.00000    0.3719    0.3719    1.0000         1
08  O          0.50000     0.00000     0.19542    0.

In [5]:
tio2.set_wyckoff_atom_parameter('Ti_2a', 'iDW', 0.35)
tio2.set_wyckoff_atom_parameter('Ti_2a', 'oDW', 0.35)
tio2.set_wyckoff_atom_parameter('O_4f', 'iDW', 0.4119)
tio2.set_wyckoff_atom_parameter('O_4f', 'oDW', 0.4119)

In [6]:
print(tio2)

Lattice:
Real space: 	[6.58069633 2.9692     6.58069633] / [90. 90. 90.]
Reciprocal space: 	[0.95479034 2.11612061 0.95479034] / [90. 90. 90.]
12 atoms in unit cell
id  Name        x/frac      y/frac      z/frac       iDW       oDW     occup  layerIdx
00  O          0.30458     0.50000     0.50000    0.4119    0.4119    1.0000         2
01  O          0.69542     0.50000     0.50000    0.4119    0.4119    1.0000         2
02  O          0.00000     0.00000     0.69542    0.4119    0.4119    1.0000         2
03  O          0.50000     0.00000     0.80458    0.4119    0.4119    1.0000         2
04  Ti         0.50000     0.00000     0.50000    0.3500    0.3500    1.0000         2
05  Ti         0.00000     0.50000     0.50000    0.3500    0.3500    1.0000         2
06  O          0.19542     0.50000     0.00000    0.4119    0.4119    1.0000         1
07  O          0.80458     0.50000     0.00000    0.4119    0.4119    1.0000         1
08  O          0.50000     0.00000     0.19542    0.

In [7]:
ruo2.set_wyckoff_atom_parameter('Ru_2a', 'iDW', 0.25)
ruo2.set_wyckoff_atom_parameter('Ru_2a', 'oDW', 0.25)
ruo2.set_wyckoff_atom_parameter('O_4f', 'iDW', 0.4119)
ruo2.set_wyckoff_atom_parameter('O_4f', 'oDW', 0.4119)

In [8]:
print(ruo2)

Lattice:
Real space: 	[6.58069633 2.9692     6.58069633] / [90. 90. 90.]
Reciprocal space: 	[0.95479034 2.11612061 0.95479034] / [90. 90. 90.]
12 atoms in unit cell
id  Name        x/frac      y/frac      z/frac       iDW       oDW     occup  layerIdx
00  O          0.31569     0.50000     0.50000    0.4119    0.4119    1.0000         2
01  O          0.68431     0.50000     0.50000    0.4119    0.4119    1.0000         2
02  O          0.00000     0.00000     0.68431    0.4119    0.4119    1.0000         2
03  O          0.50000     0.00000     0.81569    0.4119    0.4119    1.0000         2
04  Ru         0.50000     0.00000     0.50000    0.2500    0.2500    1.0000         2
05  Ru         0.00000     0.50000     0.50000    0.2500    0.2500    1.0000         2
06  O          0.18431     0.50000     0.00000    0.4119    0.4119    1.0000         1
07  O          0.81569     0.50000     0.00000    0.4119    0.4119    1.0000         1
08  O          0.50000     0.00000     0.18431    0.

In [9]:
print(tio2.translate_layered([0, 0, -2]))

Lattice:
Real space: 	[6.58069633 2.9692     6.58069633] / [90. 90. 90.]
Reciprocal space: 	[0.95479034 2.11612061 0.95479034] / [90. 90. 90.]
12 atoms in unit cell
id  Name        x/frac      y/frac      z/frac       iDW       oDW     occup  layerIdx
00  O          0.30458     0.50000    -0.50000    0.4119    0.4119    1.0000         2
01  O          0.69542     0.50000    -0.50000    0.4119    0.4119    1.0000         2
02  O          0.00000     0.00000    -0.30458    0.4119    0.4119    1.0000         2
03  O          0.50000     0.00000    -0.19542    0.4119    0.4119    1.0000         2
04  Ti         0.50000     0.00000    -0.50000    0.3500    0.3500    1.0000         2
05  Ti         0.00000     0.50000    -0.50000    0.3500    0.3500    1.0000         2
06  O          0.19542     0.50000    -1.00000    0.4119    0.4119    1.0000         1
07  O          0.80458     0.50000    -1.00000    0.4119    0.4119    1.0000         1
08  O          0.50000     0.00000    -0.80458    0.

In [10]:
tio2.addWyckoffShift?